# Sync

The `output` parameter accepts a dict with a `uri` (or `url`) field that enables syncing the local output file to a remote location (S3, GCS, etc.). Syncing can be triggered two ways:

- **Sync button** — a button in the widget UI that uploads on click
- **`ba.sync()`** — programmatic upload from the notebook

Under the hood the sync methods rely on `jupyter_bioacoustic.audio.io.write()`, using the app-configuration to set the source and destination locations.

```
Author: Brookie Guzder-Williams (bguzder-williams@berkeley.edu)
Affiliation: The Eric and Wendy Schmidt Center for Data Science & Environment
Website: https://dse.berkeley.edu/
```

In [1]:
from jupyter_bioacoustic import BioacousticAnnotator

DATA = 'data/detections.N-dse.csv'
SYNC_URI = 'https://dse-soundhub.s3.us-west-2.amazonaws.com/dev/annotations/sync-example.csv'

[JBA] Debug mode enabled. Logs → /Users/brookie/code/dse/jupyter_bioacoustic/repos/jupyter_bioacoustic/demo/jba_debug.log


---

## 1. Output Config

When `output` is a string it behaves as before — a local file path. When `output` is a dict, the following keys are available:

| Key | Type | Description |
|-----|------|-------------|
| `path` | `str` | **(required)** Local output file path — same as the string form |
| `uri` / `url` | `str` | Remote destination for sync (S3, GCS, etc.) |
| `sync_button` | `bool` or `str` | Show a sync button in the widget. `True` shows "Sync", a string sets the label. Defaults to `True` when `uri`/`url` is set |
| `recursive` | `bool` | Passed to `io.write()` — for uploading directories (e.g. partitioned parquet) |
| `secrets` | `list` | Auth kwargs for `io.write()`. Uses the same `{key, value}` format as `data_secrets` |

### Authentication

S3 auth uses boto3 defaults (env vars, `~/.aws/credentials`, IAM role). To specify a named profile or pass credentials explicitly, use the `secrets` field:

```yaml
output:
    path: outputs/results.csv
    uri: s3://my-bucket/project/results.csv
    secrets:
        - key: profile
          value: env:AWS_PROFILE      # reads $AWS_PROFILE
```

Secret values support three formats:

- `env:VAR_NAME` — reads from an environment variable
- `dialog` — prompts the user via input dialog
- any other string — used as a literal value

The secret keys are passed as kwargs to `io.write()`. For S3 the relevant keys are `profile_name`, `region_name`, and `client` (a pre-configured boto3 S3 client).

---

## 2. Sync Button

When a `uri` is configured the widget adds a sync button to the bottom-right of the form panel. Clicking it uploads the current output file, overwriting the remote copy. The button disables during upload and re-enables when complete.

The config below uses `sync_button: 'Sync to S3'` for a custom label. Set `sync_button: false` to hide the button while still allowing programmatic sync via `ba.sync()`.

In [2]:
ba = BioacousticAnnotator(
    data_path=DATA,
    sort='common_name',
    sort_order='desc',
    config='annotator_config/config/sync_example.yaml',
)

In [3]:
ba.describe()

---

**Configuration Files**

- Config: annotator_config/config/sync_example.yaml


---

**Configuration**

audio:
  column: audio_uri_public
data:
  index_column: id
  path: data/detections.N-dse.csv
data_path: data/detections.N-dse.csv
display_columns:
- deployment_code
- common_name
- scientific_name
- confidence
form_config:
  select:
    column: is_valid
    items:
    - 'yes'
    - 'no'
    label: Is [[common_name]] Valid
    required: true
  submission_buttons:
    line: true
    next:
      label: Skip
    submit:
      label: Verify
  title:
    progress_tracker: true
    value: REVIEW
output:
  index_column: detections_id
  path: outputs/sync-example.csv
  sync_button: Sync to S3
  uri: s3://dse-soundhub/dev/annotations/sync-example.csv
sort: common_name
sort_order: desc



---

In [4]:
ba.open()

<IPython.core.display.Javascript object>

In [10]:
ba.output()

,detections_id,detection_id,is_valid
0,0,0,yes
1,1,1,yes
2,2,2,no
3,3,3,yes
4,4,4,no


---

## 3. Programmatic Sync

`ba.sync()` uploads the output file to the configured `uri`. This is useful for scripted workflows, scheduled uploads, or syncing after a batch of annotations.

In [11]:
# sync to the configured uri (s3://my-bucket/project/annotations/sync-example.csv)
ba.sync()

's3://dse-soundhub/dev/annotations/sync-example.csv'

You can override the destination or pass additional auth kwargs:

In [12]:
# override destination
ba.sync(dest='s3://dse-soundhub/dev/annotations/sync-example-2.csv')

's3://dse-soundhub/dev/annotations/sync-example-2.csv'

In [14]:
# override auth
ba.sync(profile='prod', region_name='us-east-1')

's3://dse-soundhub/dev/annotations/sync-example.csv'

For more control use `io.write()` directly — `ba.sync()` is a convenience wrapper around it:

In [24]:
from jupyter_bioacoustic.audio import io
help(io.write)

Help on function write in module jupyter_bioacoustic.audio.io:

write(src: 'str', dest: 'str', recursive: 'bool' = False, overwrite: 'bool' = True, **kwargs: 'Any') -> 'str'
    Write a file or directory to any destination.

    Returns:
        Destination path/URI (str).



In [25]:
io.write(
    'outputs/sync-example.csv',
    's3://dse-soundhub/dev/annotations/sync-example-3.csv',
)

's3://dse-soundhub/dev/annotations/sync-example-3.csv'